# AprilTag PnP Stability Test
Measure multi-frame PnP translation and orientation stability. A valid camera calibration YAML is mandatory; this notebook never falls back to DepthNet.

In [ ]:
from __future__ import print_function
import os, sys, time, traceback
import cv2, numpy as np, yaml, ipywidgets as widgets
from IPython.display import display
current=os.path.abspath(os.getcwd())
while not os.path.isfile(os.path.join(current,'config.json')):
    parent=os.path.dirname(current)
    if parent==current: raise RuntimeError('project root not found')
    current=parent
PROJECT_ROOT=current
if PROJECT_ROOT not in sys.path: sys.path.insert(0,PROJECT_ROOT)
from demo_core import load_config
from demo_core.perception import AprilTagBinDetector, DepthSensor
from tuning_tools.diagnostic_tools import append_csv, pose_from_tag_corners, summarize_pose_samples, timestamped_log_path

state={'camera':None,'detector':None,'matrix':None,'dist':None,'csv':None}
camera_real=widgets.Checkbox(value=False,description='camera_real')
yaml_path=widgets.Text(value=os.path.join(PROJECT_ROOT,'assets','calibration','jetbot_camera_320x240.yaml'),description='calibration',layout=widgets.Layout(width='650px'))
marker_m=widgets.FloatText(value=0.08,description='marker_m')
frames=widgets.IntText(value=100,description='frames')
sample_hz=widgets.FloatText(value=5.0,description='sample_hz')
known_z=widgets.FloatText(value=0.0,description='known_z_m')
scenario=widgets.Text(value='front_center',description='scenario')
image=widgets.Image(format='jpeg',width=640,height=480)
output=widgets.Output(layout={'border':'1px solid #bbb','height':'340px','overflow_y':'auto'})

def start(_=None):
    with output:
        try:
            path=os.path.abspath(yaml_path.value)
            if not os.path.isfile(path): raise RuntimeError('calibration YAML does not exist: '+path)
            with open(path,'r') as stream: calib=yaml.safe_load(stream)
            state['matrix']=np.asarray(calib['camera_matrix'],dtype=np.float32); state['dist']=np.asarray(calib['dist_coeff'],dtype=np.float32)
            if state['matrix'].shape!=(3,3): raise RuntimeError('camera_matrix must be 3x3')
            if not camera_real.value: raise RuntimeError('enable camera_real before starting')
            cfg=load_config(overrides={'runtime':{'dry_run':{'camera':False}},'camera':{'calibration_yaml':path},'detectors':{'bin':{'marker_length_m':float(marker_m.value)}}})
            state['camera']=DepthSensor(cfg); state['camera'].start(camera_only=True); state['detector']=AprilTagBinDetector(cfg); state['detector'].load()
            state['csv']=timestamped_log_path(PROJECT_ROOT,'apriltag_pnp','samples.csv'); print('camera + tag ready')
        except Exception: traceback.print_exc()

def run(_=None):
    with output:
        try:
            if state['camera'] is None: raise RuntimeError('start first')
            samples=[]; attempted=int(frames.value)
            for index in range(attempted):
                frame=state['camera'].read_frame(); observation=state['detector'].detect(frame); canvas=frame.copy()
                if observation.get('found') and len(observation.get('corners',[]))==4:
                    pose=pose_from_tag_corners(observation['corners'],state['matrix'],state['dist'],float(marker_m.value)); pose.update({'timestamp':time.time(),'frame_index':index,'scenario':scenario.value})
                    if float(known_z.value)>0.0: pose['known_z_error_m']=pose['perpendicular_distance_m']-float(known_z.value)
                    samples.append(pose); pts=np.asarray(observation['corners'],dtype=np.int32); cv2.polylines(canvas,[pts],True,(0,255,255),2); cv2.putText(canvas,'z={:.3f} yaw={:.1f}'.format(pose['z_m'],pose['yaw_deg']),(10,25),cv2.FONT_HERSHEY_SIMPLEX,0.55,(0,255,255),2)
                ok,encoded=cv2.imencode('.jpg',cv2.resize(canvas,(640,480))); image.value=encoded.tobytes() if ok else b''
                time.sleep(1.0/max(0.5,float(sample_hz.value)))
            append_csv(state['csv'],samples); print('summary',summarize_pose_samples(samples,attempted)); print('csv',state['csv'])
        except Exception: traceback.print_exc()

def release(_=None):
    if state['camera'] is not None: state['camera'].stop()
    state.update({'camera':None,'detector':None}); output.append_stdout('camera released\n')
buttons=[]
for label,fn,style in [('Load Calibration + Start',start,'info'),('Run Stability Test',run,'success'),('STOP + Release',release,'danger')]:
    button=widgets.Button(description=label,button_style=style); button.on_click(fn); buttons.append(button)
display(widgets.VBox([camera_real,yaml_path,widgets.HBox([marker_m,frames,sample_hz,known_z,scenario]),widgets.HBox(buttons),image,output]))
